# PokouAI — Train cocoa-disease CNN with Grad-CAM

**Purpose**: a fast, small, accurate closed-set classifier that runs on any phone with no LLM gymnastics — as a parallel/fallback path to the Gemma 4 fine-tune. The LLM generates the *explanation* (symptoms / treatment / prevention text); the CNN provides the *label*.

**Architecture**: EfficientNet-B0, ImageNet-pretrained, head replaced for 6 cocoa disease classes (matches `cocoa_diseases.json`):

- `black_pod`, `frosty_pod_rot`, `swollen_shoot`, `vascular_streak_dieback`, `healthy`, `other_damage`

**Mobile artefact**: TFLite (universal — `react-native-fast-tflite`) + ONNX (alternative — `onnxruntime-react-native`). Expected size: ~5-15 MB at fp16. Runs <100 ms / image on iPhone, faster on GPU/NPU.

**Grad-CAM**: visualizes which image regions the model attended to. Built with forward/backward hooks on the last conv block — no extra dependency. Heatmaps render only at training time (gradient computation isn't usually available in mobile inference runtimes). On-device interpretability later can use plain CAM (gradient-free) since EfficientNet is GAP-headed.

**Prereq**: training data prepared via the existing pipeline (`prepare_training_data.py` → `augment_images.py`) or attached as a Kaggle Dataset. The dataset must be an ImageFolder layout: `<root>/<class_name>/*.jpg`.

## 0 — Parameters

In [ ]:
# Path to the Kaggle Dataset (or local dir). The probe below finds the
# directory whose children are class folders and trains on whatever subset
# of CANONICAL_CLASSES is actually present (missing classes are warned, not
# fatal — the CNN trains on what you have).
DATA_BASE = '/kaggle/input/datasets/yaokouadio/pokou-ai-cocoa-training-data'

# Output dir — checkpoints, exported artefacts, Grad-CAM viz.
OUT_DIR = '/kaggle/working/cnn_v1'

# Canonical class list (matches cocoa_diseases.json). The trained model's
# output is whichever subset of these is actually present in the dataset.
CANONICAL_CLASSES = [
    'black_pod',
    'frosty_pod_rot',
    'swollen_shoot',
    'vascular_streak_dieback',
    'healthy',
    'other_damage',
]

# Training knobs. Defaults tuned for ~2-3k images per class on a single T4.
IMG_SIZE   = 224          # EfficientNet-B0 native input
BATCH_SIZE = 32
EPOCHS     = 15
LR         = 1e-4         # AdamW; head fine-tunes fast, backbone slowly
VAL_SPLIT  = 0.15         # 70/15/15 train/val/test
TEST_SPLIT = 0.15
SEED       = 42
PATIENCE   = 3            # early stop after N epochs without val_acc improvement
MIN_DELTA  = 1e-3         # val_acc gain below this counts as "no improvement"

import os, json, time, random
import numpy as np
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

# Find the dir whose immediate children include the most canonical classes.
# Searches the base + common subdir patterns.
def find_imagefolder_root(base: str, canonical: list[str]) -> tuple[str, list[str]]:
    candidates = [
        base,
        os.path.join(base, 'processed', 'images'),
        os.path.join(base, 'images'),
        os.path.join(base, 'processed'),
    ]
    best_root, best_present = None, []
    for c in candidates:
        if not os.path.isdir(c):
            continue
        present = [cls for cls in canonical if os.path.isdir(os.path.join(c, cls))]
        if len(present) > len(best_present):
            best_root, best_present = c, present
    if not best_root or len(best_present) < 2:
        print(f'Walked {base}; found < 2 canonical class dirs anywhere. Contents:')
        for root, dirs, _ in os.walk(base):
            depth = root[len(base):].count(os.sep)
            if depth <= 2:
                print(f'  {"  " * depth}{os.path.basename(root) or root}/  ({len(dirs)} subdirs)')
        raise FileNotFoundError(
            f'Could not locate at least 2 canonical class dirs under {base}.'
        )
    return best_root, best_present

DATA_ROOT, CLASSES = find_imagefolder_root(DATA_BASE, CANONICAL_CLASSES)
missing = [c for c in CANONICAL_CLASSES if c not in CLASSES]

print(f'image root : {DATA_ROOT}')
print(f'classes    : {len(CLASSES)} of {len(CANONICAL_CLASSES)} canonical')
for c in CLASSES:
    p = os.path.join(DATA_ROOT, c)
    n = len([f for f in os.listdir(p) if not f.startswith('.')])
    print(f'  ✓ {c:30s} {n:6d} images')
for c in missing:
    print(f'  ✗ {c:30s} (not in dataset — model won\'t predict this label)')

if missing:
    print(f'\n⚠ Training on {len(CLASSES)} classes. The app\'s cocoa_diseases.json '
          f'still defines all {len(CANONICAL_CLASSES)}, but the CNN can only output a '
          f'class it has training data for. {", ".join(missing)} will need to be '
          f'handled by the LLM or routed to "uncertain" in the app.')

## 1 — Install

Most of these are on the Kaggle base image already; `ai-edge-torch` is the only meaningful addition for the TFLite export step.

In [ ]:
!pip install -q --no-cache-dir torchvision scikit-learn matplotlib seaborn ai-edge-torch onnx onnxruntime

import torch
print(f'torch         : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device   : {torch.cuda.get_device_name(0)}')

## 2 — Data

Standard ImageFolder + train/val/test split + augmentations. The `CLASSES` order is forced via a custom class-to-index map so the exported model's output indices match `cocoa_diseases.json` ordering and the app can do `CLASSES[argmax]` without remapping.

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

# ImageNet normalization — matches the pretrained backbone weights.
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Build the base ImageFolder, then force class ordering to match CLASSES.
base = datasets.ImageFolder(DATA_ROOT)
name_to_idx = {c: i for i, c in enumerate(CLASSES)}
missing = set(CLASSES) - set(base.class_to_idx)
assert not missing, f'missing class dirs in {DATA_ROOT}: {missing}'
# Remap so the exported model's labels match CLASSES ordering exactly.
base.samples = [(path, name_to_idx[base.classes[idx]]) for path, idx in base.samples]
base.targets = [t for _, t in base.samples]
base.classes = list(CLASSES)
base.class_to_idx = name_to_idx

# Stratified 70/15/15 split — keeps class ratios consistent across splits,
# matters when a class (e.g. healthy) is over-represented in raw data.
indices = list(range(len(base)))
targets = base.targets
train_idx, temp_idx, _, temp_y = train_test_split(
    indices, targets, test_size=VAL_SPLIT + TEST_SPLIT, stratify=targets, random_state=SEED,
)
val_size = VAL_SPLIT / (VAL_SPLIT + TEST_SPLIT)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=1 - val_size, stratify=temp_y, random_state=SEED,
)

# Same ImageFolder, two transform views via Subset wrappers.
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, ds, indices, tf):
        self.ds, self.indices, self.tf = ds, indices, tf
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, y = self.ds.samples[self.indices[i]]
        from PIL import Image
        return self.tf(Image.open(img).convert('RGB')), y

train_ds = TransformedSubset(base, train_idx, train_tf)
val_ds   = TransformedSubset(base, val_idx,   eval_tf)
test_ds  = TransformedSubset(base, test_idx,  eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'train : {len(train_ds)}')
print(f'val   : {len(val_ds)}')
print(f'test  : {len(test_ds)}')

## 3 — Model + training

EfficientNet-B0 with ImageNet weights, classifier head replaced with `Linear(features, 6)`. AdamW + cosine LR schedule. Saves the best checkpoint on val accuracy.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def build_model() -> nn.Module:
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    # Replace the 1000-class classifier with a 6-class one.
    in_features = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(in_features, len(CLASSES))
    return m

model = build_model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, train_mode: bool):
    model.train(train_mode)
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if train_mode:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train_mode):
            logits = model(x)
            loss = criterion(logits, y)
            if train_mode:
                loss.backward()
                optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        total    += x.size(0)
    return loss_sum / total, correct / total

best_val_acc = 0.0
best_path = f'{OUT_DIR}/cocoa_cnn_best.pth'
history = []
epochs_no_improve = 0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step()
    history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc, 'val_loss': val_loss, 'val_acc': val_acc})
    print(f'epoch {epoch:>2}/{EPOCHS}  tr_loss={tr_loss:.3f} tr_acc={tr_acc:.3f}  val_loss={val_loss:.3f} val_acc={val_acc:.3f}  ({time.time()-t0:.0f}s)')
    if val_acc > best_val_acc + MIN_DELTA:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save({
            'state_dict': model.state_dict(),
            'classes': CLASSES,
            'img_size': IMG_SIZE,
            'val_acc': val_acc,
            'epoch': epoch,
        }, best_path)
        print(f'  ✓ best saved → {best_path} (val_acc={val_acc:.3f})')
    else:
        epochs_no_improve += 1
        print(f'  · no improvement ({epochs_no_improve}/{PATIENCE})')
        if epochs_no_improve >= PATIENCE:
            print(f'  ⏹ early stop — no val_acc gain ≥{MIN_DELTA} for {PATIENCE} epochs (best={best_val_acc:.3f})')
            break

with open(f'{OUT_DIR}/history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'\ntraining complete — best val_acc={best_val_acc:.3f}')

## 4 — Evaluate on test set

Per-class accuracy + confusion matrix. The test split was held out from training — these numbers are the honest signal.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Restore best weights.
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        preds = model(x).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.numpy())

print(classification_report(all_true, all_preds, target_names=CLASSES, digits=3))

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=CLASSES, yticklabels=CLASSES, cbar=False)
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('confusion matrix (test set)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=120)
plt.show()

## 5 — Grad-CAM

What did the network actually look at? Hook the last conv block's forward output + backward gradient w.r.t. the predicted class, weight the activations by the channel-wise gradient mean, ReLU, upsample to image size, overlay.

Gradient-free CAM is feasible later for on-device inference (EfficientNet-B0 is GAP-headed). For now this is for the report / sanity-check that the model attended to the lesion, not the background.

In [ ]:
import numpy as np
import torch.nn.functional as F
from PIL import Image

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x: torch.Tensor, class_idx=None):
        self.model.eval()
        out = self.model(x)
        if class_idx is None:
            class_idx = int(out.argmax(1).item())
        self.model.zero_grad()
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        probs = F.softmax(out, dim=1).squeeze().detach().cpu().numpy()
        return cam, class_idx, probs

# EfficientNet-B0: last conv block is `features[-1]` (the final MBConv stack).
cam_extractor = GradCAM(model, target_layer=model.features[-1])

# Visualize 2 examples per class.
def denorm(t: torch.Tensor) -> np.ndarray:
    arr = t.detach().cpu().numpy().transpose(1, 2, 0)
    arr = arr * np.array(STD) + np.array(MEAN)
    return np.clip(arr, 0, 1)

rows, cols = len(CLASSES), 4
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
for r, cls_name in enumerate(CLASSES):
    # Pull first 2 test images of this class.
    cls_idx = CLASSES.index(cls_name)
    samples = [(base.samples[i][0], base.samples[i][1]) for i in test_idx if base.samples[i][1] == cls_idx][:2]
    for c in range(2):
        if c >= len(samples):
            axes[r, c*2].axis('off'); axes[r, c*2+1].axis('off'); continue
        img_path, _ = samples[c]
        img = Image.open(img_path).convert('RGB')
        x = eval_tf(img).unsqueeze(0).to(device)
        cam, pred_idx, probs = cam_extractor(x)
        img_np = denorm(x.squeeze())
        ax_img = axes[r, c*2]; ax_img.imshow(img_np); ax_img.set_title(f'{cls_name}', fontsize=9); ax_img.axis('off')
        ax_cam = axes[r, c*2+1]
        ax_cam.imshow(img_np); ax_cam.imshow(cam, cmap='jet', alpha=0.45)
        pred_name = CLASSES[pred_idx]
        ok = '✓' if pred_idx == cls_idx else '✗'
        ax_cam.set_title(f'{ok} {pred_name} ({probs[pred_idx]:.2f})', fontsize=9)
        ax_cam.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/gradcam_examples.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✓ saved → {OUT_DIR}/gradcam_examples.png')

## 6 — Export for mobile

**ONNX** — universal, loads with `onnxruntime-react-native` on iOS + Android. ~20 MB FP32, ~10 MB FP16.

**TFLite** via `ai-edge-torch` — smaller (int8 quantized ~5 MB), loads with `react-native-fast-tflite`. Same conversion path as the Gemma 4 fine-tune used in notebook 04, minus the LLM-specific generative bits.

In [ ]:
import torch, os

# Make sure the model is in eval mode and on CPU for export.
model.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

# --- ONNX (FP32) ---
onnx_path = f'{OUT_DIR}/cocoa_cnn_v1.onnx'
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)
print(f'✓ ONNX written → {onnx_path} ({os.path.getsize(onnx_path)/1e6:.2f} MB)')

# Sanity check the ONNX matches PyTorch logits within float noise.
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
torch_out = model(dummy).detach().numpy()
ort_out = sess.run(None, {'image': dummy.numpy()})[0]
max_diff = float(np.abs(torch_out - ort_out).max())
print(f'  ONNX vs PyTorch max abs diff: {max_diff:.6f}')
assert max_diff < 1e-3, 'ONNX export diverges from PyTorch — check opset / dynamic_axes'

# --- TFLite via ai-edge-torch ---
# Works for image classifiers without the gemma4 generative example issues —
# the CNN graph has no static KV cache or PLE memory.
tflite_path = f'{OUT_DIR}/cocoa_cnn_v1.tflite'
try:
    import ai_edge_torch
    edge_model = ai_edge_torch.convert(model, (dummy,))
    edge_model.export(tflite_path)
    print(f'✓ TFLite written → {tflite_path} ({os.path.getsize(tflite_path)/1e6:.2f} MB)')
except Exception as e:
    print(f'⚠ TFLite export skipped: {type(e).__name__}: {e}')
    print('  Fall back to ONNX on-device, or rerun ai-edge-torch from github main.')

# Sidecar metadata — the app reads this to map argmax → class name.
meta = {
    'model_version': 'cocoa_cnn_v1',
    'classes': CLASSES,
    'img_size': IMG_SIZE,
    'normalize_mean': MEAN,
    'normalize_std': STD,
    'best_val_acc': best_val_acc,
    'epoch': ckpt.get('epoch'),
    'architecture': 'efficientnet_b0',
    'pretrained': 'imagenet1k_v1',
}
with open(f'{OUT_DIR}/cocoa_cnn_v1.meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'✓ metadata    → {OUT_DIR}/cocoa_cnn_v1.meta.json')
!ls -lh {OUT_DIR}

## 7 — Notes

**Where this fits**: parallel to the Gemma 4 fine-tune. The CNN gives a fast, accurate label that any phone can run; the LLM (on-device or hub) generates the symptom / treatment / prevention text given that label + the image.

**On-device integration (React Native)**:

- **TFLite (recommended for size)** — install `react-native-fast-tflite`; load `cocoa_cnn_v1.tflite`; preprocess image to 224×224, normalize with `MEAN`/`STD` from the meta JSON; output is `[1, 6]` logits → `argmax` → `CLASSES[i]`. Add a softmax for the confidence score.
- **ONNX (recommended for portability)** — install `onnxruntime-react-native`; same preprocessing; same output handling.

**Integration in `InferenceRouter.ts`**: add a `'cnn'` tier above `'local'` LLM. CNN runs first (fast, deterministic label); LLM is asked to *explain* the CNN's label rather than to diagnose from scratch. Cuts LLM token budget and improves answer quality.

**On-device interpretability (v1.1)**: Grad-CAM needs autograd which isn't on most mobile runtimes. EfficientNet-B0 has a GAP head, so plain CAM (just the last conv activations × classifier weights, no gradients) works on inference-only runtimes — implement in JS once we have the tflite/onnx pipeline shipping.

**If accuracy is weaker than expected**:

- Look at the confusion matrix from §4 — which classes are getting confused? Cocoa diseases that share black-rot symptoms (`black_pod` ↔ `frosty_pod_rot`) are the usual culprits.
- More training data is the highest-leverage fix; augmentation can't replace true diversity.
- Consider class weighting in `CrossEntropyLoss` if a class is sparse (`weight=` arg).
- Try EfficientNet-B1 (~2× params, ~1.5% accuracy bump) before reaching for heavier backbones.